# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# imports

import os
import json
import inspect
import builtins
import importlib
from openai import OpenAI
import gradio as gr

## Setup: connect to local Ollama

No API keys needed - everything runs through a local Ollama server using the OpenAI-compatible endpoint. Before running this notebook, make sure Ollama is running and you've pulled both models:

```
ollama pull gemma4:e4b-mlx
ollama pull llama3.2
```

In [2]:
# Constants

OLLAMA_URL = "http://localhost:11434/v1"

MODEL_GEMMA = "gemma4:e4b-mlx"
MODEL_LLAMA = "gemma4:26b-mlx"
MODELS = [MODEL_GEMMA, MODEL_LLAMA]

ollama = OpenAI(base_url=OLLAMA_URL, api_key="ollama")

## System prompt: give the assistant an area of technical expertise

The area of expertise picked in the UI is folded into the system prompt, so the same tool can act as a focused tutor for different topics.

In [3]:
EXPERTISE_PROMPTS = {
    "General Programming": "You are a patient, expert software engineering tutor. \
A student will ask you a technical question - it might be about a snippet of code, a concept, \
or general programming practice. Explain the answer clearly and in depth, including *why* it works, \
not just what it does. Use plain language and a short example where it helps. Respond in markdown.",

    "Python": "You are a patient, expert Python tutor. \
A student will ask you a technical question about Python - a snippet of code, a standard library function, \
or a language feature. Explain what it does, why it's written that way, and mention any gotchas. \
Use plain language and a short example where it helps. Respond in markdown.",

    "Data Science": "You are a patient, expert Data Science tutor, comfortable with pandas, numpy and ML concepts. \
A student will ask you a technical question. Explain the answer clearly, including *why* it works, \
and how it fits into a typical data science workflow. Respond in markdown.",

    "Web Development": "You are a patient, expert web development tutor, comfortable with JavaScript, HTML, CSS and APIs. \
A student will ask you a technical question. Explain the answer clearly, including *why* it works, \
and any browser or framework quirks worth knowing. Respond in markdown.",
}

TOOL_HINT = "\n\nIf you need an authoritative definition of a Python built-in, keyword or standard library \
function, call the get_python_doc tool rather than guessing."

## Bonus: a tool the assistant can call

`get_python_doc` looks up the real docstring for a Python built-in or a name from a small allowed set of standard library modules, so the assistant can ground an answer in the actual documentation instead of guessing.

In [4]:
# A safe, read-only tool: look up the real docstring for a Python built-in or standard library name
# e.g. "zip", "str.split", "itertools.chain", "os.path.join"

ALLOWED_MODULES = ["itertools", "collections", "functools", "os", "sys", "re", "json", "math", "datetime"]

def get_python_doc(identifier: str) -> str:
    print(f"Tool called: get_python_doc('{identifier}')", flush=True)
    head, *rest = identifier.split(".")

    obj = getattr(builtins, head, None)
    if obj is None and head in ALLOWED_MODULES:
        obj = importlib.import_module(head)

    if obj is None:
        return f"No documentation found for '{identifier}'"

    try:
        for part in rest:
            obj = getattr(obj, part)
    except AttributeError:
        return f"No documentation found for '{identifier}'"

    doc = inspect.getdoc(obj)
    return doc if doc else f"No documentation found for '{identifier}'"

In [5]:
get_python_doc_function = {
    "name": "get_python_doc",
    "description": "Look up the real docstring for a Python built-in, or a name from a small set of \
standard library modules (itertools, collections, functools, os, sys, re, json, math, datetime). \
Use this to check exact behavior instead of guessing.",
    "parameters": {
        "type": "object",
        "properties": {
            "identifier": {
                "type": "string",
                "description": "A dotted Python identifier to look up, e.g. 'zip', 'str.split' or 'itertools.chain'",
            },
        },
        "required": ["identifier"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": get_python_doc_function}]

In [6]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_python_doc":
            arguments = json.loads(tool_call.function.arguments)
            result = get_python_doc(arguments.get("identifier", ""))
        else:
            result = f"Unknown tool: {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

## Chat function: resolve any tool calls, then stream the final answer

The first call is non-streaming so we can check `finish_reason` and resolve tool calls in a loop. Once the model is done calling tools, we re-issue the same conversation with `stream=True` so the final answer appears incrementally in the UI.

In [7]:
def chat(message, history, model, expertise):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    system_message = EXPERTISE_PROMPTS[expertise] + TOOL_HINT
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = ollama.chat.completions.create(model=model, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses = handle_tool_calls(tool_message)
        messages.append(tool_message)
        messages.extend(tool_responses)
        response = ollama.chat.completions.create(model=model, messages=messages, tools=tools)

    stream = ollama.chat.completions.create(model=model, messages=messages, stream=True)
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

## Gradio UI: pick a model and an area of expertise, then chat

In [8]:
model_selector = gr.Dropdown(MODELS, value=MODEL_GEMMA, label="Model")
expertise_selector = gr.Dropdown(list(EXPERTISE_PROMPTS.keys()), value="Python", label="Area of expertise")

view = gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_selector, expertise_selector],
    title="Technical Question Answerer",
    examples=[
        ["Please explain what this code does and why:\nyield from {book.get('author') for book in books if book.get('author')}", MODEL_GEMMA, "Python"],
        ["What's the difference between a list and a generator in Python?", MODEL_GEMMA, "Python"],
    ],
)

view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: get_python_doc('str.split')
